# ABSA app reviews collector

Gets the latest Absa Banking app reviews from the Apple App Store and Google Play (South Africa), combines them into one table and saves them to `data/reviews.csv`.

Each run adds new reviews and removes duplicates, so you can run it again at any time.

**One-time setup:** `pip install google-play-scraper` (in the same Python that runs this notebook).

## 1. Settings

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from google_play_scraper import Sort, reviews

APPLE_APP_ID = "1085620596"                  # Absa Banking (App Store)
GOOGLE_APP_ID = "com.barclays.absa.banking"   # Absa Banking App (Google Play)
COUNTRY = "za"
GOOGLE_MAX_REVIEWS = 1000                     # newest N reviews per run

DATA_FILE = Path("data/reviews.csv")
DATA_FILE.parent.mkdir(exist_ok=True)

COLUMNS = ["store", "review_id", "rating", "title", "text", "app_version",
           "review_date", "thumbs_up", "reply_text", "reply_date", "fetched_at"]

## 2. Apple App Store

Apple's public RSS feed gives up to 10 pages of 50 reviews (about 500 newest reviews).

In [ ]:
def fetch_apple_reviews(app_id=APPLE_APP_ID, country=COUNTRY, max_pages=10):
    rows = []
    for page in range(1, max_pages + 1):
        url = (f"https://itunes.apple.com/{country}/rss/customerreviews/"
               f"page={page}/id={app_id}/sortby=mostrecent/json")
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        entries = resp.json().get("feed", {}).get("entry", [])
        if isinstance(entries, dict):   # a page with one entry is not a list
            entries = [entries]
        entries = [e for e in entries if "im:rating" in e]
        if not entries:
            break
        for e in entries:
            rows.append({
                "store": "apple",
                "review_id": e["id"]["label"],
                "author": e["author"]["name"]["label"],
                "rating": int(e["im:rating"]["label"]),
                "title": e["title"]["label"],
                "text": e["content"]["label"],
                "app_version": e["im:version"]["label"],
                "review_date": pd.to_datetime(e["updated"]["label"], utc=True),
                "thumbs_up": int(e.get("im:voteSum", {}).get("label", 0)),
                "reply_text": None,   # not in Apple's public feed
                "reply_date": None,
            })
    return pd.DataFrame(rows)

apple_df = fetch_apple_reviews()
print(f"Apple: {len(apple_df)} reviews")
apple_df.head()

## 3. Google Play

Uses the `google-play-scraper` library. This is not an official Google API, so it can break when Google changes its site.

In [ ]:
def fetch_google_reviews(app_id=GOOGLE_APP_ID, country=COUNTRY, max_reviews=GOOGLE_MAX_REVIEWS):
    results, token = [], None
    while len(results) < max_reviews:
        batch, token = reviews(app_id, lang="en", country=country, sort=Sort.NEWEST,
                               count=min(200, max_reviews - len(results)),
                               continuation_token=token)
        results.extend(batch)
        if not batch or token is None or token.token is None:
            break
    rows = [{
        "store": "google",
        "review_id": r["reviewId"],
        "author": r["userName"],
        "rating": r["score"],
        "title": None,   # Google reviews have no title
        "text": r["content"],
        "app_version": r.get("reviewCreatedVersion") or r.get("appVersion"),
        "review_date": pd.to_datetime(r["at"]).tz_localize("UTC"),
        "thumbs_up": r["thumbsUpCount"],
        "reply_text": r.get("replyContent"),
        "reply_date": pd.to_datetime(r["repliedAt"]).tz_localize("UTC") if r.get("repliedAt") else None,
    } for r in results]
    return pd.DataFrame(rows)

google_df = fetch_google_reviews()
print(f"Google: {len(google_df)} reviews")
google_df.head()

## 4. Combine and save

New reviews are added to `data/reviews.csv`. If a review already exists (same store and ID), the newest copy is kept, because people can edit their reviews.

In [ ]:
new_df = pd.concat([apple_df, google_df], ignore_index=True)
new_df["fetched_at"] = datetime.now(timezone.utc)
new_df = new_df[COLUMNS]

if DATA_FILE.exists():
    old_df = pd.read_csv(DATA_FILE, dtype={"review_id": str})
    all_df = pd.concat([old_df, new_df.astype({"review_id": str})], ignore_index=True)
else:
    old_df = pd.DataFrame(columns=COLUMNS)
    all_df = new_df

all_df = all_df.drop_duplicates(subset=["store", "review_id"], keep="last")
all_df.to_csv(DATA_FILE, index=False)

added = len(all_df) - len(old_df)
print(f"Saved {len(all_df)} reviews to {DATA_FILE} ({added} new)")

## 5. Quick check

In [ ]:
all_df["review_date"] = pd.to_datetime(all_df["review_date"], utc=True, format="mixed")

summary = all_df.groupby("store").agg(
    reviews=("review_id", "count"),
    avg_rating=("rating", "mean"),
    oldest=("review_date", "min"),
    newest=("review_date", "max"),
).round(2)
summary

In [ ]:
# Star split per store (% of reviews)
(pd.crosstab(all_df["store"], all_df["rating"], normalize="index") * 100).round(1)